# RAGAS Evaluation — Colab Notebook

This notebook contains the **evaluation stage** of the RAG project.

The objective was to compare four retrieval configurations using **RAGAS**:

- 2 embedding models
- 2 chunking strategies
- 4 RAG configurations in total
- Metrics: Faithfulness, Answer Relevancy, Context Precision, and Context Recall

The final comparison is based on the evaluation results generated and saved during the Colab runs.

## Why Google Colab?

The RAGAS evaluation required repeated inference through the Prometheus evaluator model.  
The local machine could not process the evaluation efficiently, so the evaluation work was shifted to **Google Colab with GPU support**.

Model files and evaluation data were stored in Google Drive so they could be reused across Colab sessions.

## 1. Colab Environment & Model Setup

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU is active: {torch.cuda.get_device_name(0)}")
else:
    print(" CPU is active (No GPU detected)")

✅ GPU is active: Tesla T4


In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/RAG_Project/models"

print(os.listdir(MODEL_DIR))

['prometheus-7b-v2.0', 'bge-base-en-v1.5', 'prometheus-7b-v2.0-GGUF']


### Local model copies

For faster inference, the Prometheus GGUF model and the BGE embedding model were copied from Google Drive to Colab's local storage.

In [ ]:
import os
import shutil
import subprocess
import time

PROMETHEUS_GGUF = "/content/drive/MyDrive/RAG_Project/models/prometheus-7b-v2.0-GGUF/prometheus-7b-v2.0.Q4_K_M.gguf"

LOCAL_PROMETHEUS = "/content/prometheus-7b-v2.0.Q4_K_M.gguf"

if not os.path.exists(LOCAL_PROMETHEUS):
    print("Copying Prometheus...")
    shutil.copy2(PROMETHEUS_GGUF, LOCAL_PROMETHEUS)
    print("Copy complete ✅")
else:
    print("Model already exists locally ✅")

Copying Prometheus...
Copy complete ✅


In [ ]:
BGE_LOCAL = "/content/bge-base-en-v1.5"

In [ ]:
import os
import shutil

if not os.path.exists(BGE_LOCAL):
    print("Copying BGE...")
    shutil.copytree(BGE_PATH, BGE_LOCAL)
    print("BGE copy complete ✅")
else:
    print("BGE already exists locally ✅")

Copying BGE...
BGE copy complete ✅


### Prometheus evaluation server

Prometheus was served locally through the OpenAI-compatible `llama-cpp` API so that RAGAS could use it as the evaluator LLM.

In [ ]:
import subprocess
import sys
import time
import os

server_log = open("/content/llama_server.log", "w")

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "llama_cpp.server",
        "--model", LOCAL_PROMETHEUS,
        "--model_alias", "prometheus",
        "--n_gpu_layers", "-1",
        "--n_ctx", "8192",
        "--host", "127.0.0.1",
        "--port", "8000"
    ],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(8)

print("Server status:", server.poll())

Server status: None


In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/v1/models"
)

print(response.status_code)
print(response.json())

200
{'object': 'list', 'data': [{'id': 'prometheus', 'object': 'model', 'owned_by': 'me', 'permissions': []}]}


### API smoke test

In [ ]:
from openai import AsyncOpenAI

client = AsyncOpenAI(
    api_key="anything",
    base_url="http://127.0.0.1:8000/v1"
)

In [ ]:
import time

start = time.time()

response = await client.chat.completions.create(
    model="prometheus",
    messages=[
        {
            "role": "user",
            "content": "What is Retrieval-Augmented Generation?"
        }
    ],
    temperature=0,
    max_tokens=100
)

print(response.choices[0].message.content)

print("\nTime:", round(time.time() - start, 2), "seconds")

Retrieval-Augmented Generation (RAG) is a method that combines the strengths of both retrieval and generation processes. It involves using information from external sources to enhance the quality of generated content. This approach allows for more accurate and contextually relevant responses, as it leverages existing knowledge bases or databases to inform the generation process. The integration of retrieval with generation is a significant advancement in the field of natural language processing, as it enables systems to provide more nuanced

Time: 2.84 seconds


## 2. RAGAS Setup

RAGAS was used with **Prometheus-7B-v2.0** as the evaluator LLM.

The BGE model below is used by RAGAS for Answer Relevancy evaluation and is kept fixed across experiments.

In [ ]:
!pip install -q \
    ragas==0.4.3 \
    "langchain-community==0.4.1" \
    openai \
    sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

In [ ]:
import ragas

print(ragas.__version__)

0.4.3


In [ ]:
from ragas.llms import llm_factory

evaluator_llm = llm_factory(
    "prometheus",
    client=client,
    provider="openai"
)

In [ ]:
from ragas.embeddings import HuggingFaceEmbeddings

ragas_embeddings = HuggingFaceEmbeddings(
    model=BGE_LOCAL,
    use_api=False
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

faithfulness = Faithfulness(
    llm=evaluator_llm
)

answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=ragas_embeddings
)

context_precision = ContextPrecision(
    llm=evaluator_llm
)

context_recall = ContextRecall(
    llm=evaluator_llm
)

## 3. Evaluation Dataset

The four experiment outputs were loaded from the saved JSON file.  
Ground-truth answers were loaded from the evaluation CSV and attached to each generated result.

In [ ]:
import json

EXPERIMENT_FILE = "/content/drive/MyDrive/RAG_Project/data/experiments.json"

with open(EXPERIMENT_FILE, "r", encoding="utf-8") as f:
    all_exp = json.load(f)

print(all_exp.keys())

dict_keys(['emb_a_semantic', 'emb_b_semantic', 'emb_a_recursive', 'emb_b_recursive'])


In [ ]:
GT_FILE = "/content/drive/MyDrive/RAG_Project/data/rag_eval_qa_dataset.csv"

que = pd.read_csv(GT_FILE)

que.head()

,id,question,ground_truth_answer,source_document,topic,difficulty,question_type
0,q01,"When was ChatGPT launched, and which underlyin...","ChatGPT launched on November 30, 2022. It is p...",LLM_survey.pdf,model timeline,easy,factual
1,q02,How many parameters does the flagship GPT-3 mo...,175 billion parameters. GPT-3 is considered th...,LLM_survey.pdf,scaling / emergent abilities,easy,numeric
2,q03,What are the three LLM families the survey foc...,"GPT (released by OpenAI), LLaMA (released by M...",LLM_survey.pdf,LLM families,easy,factual
3,q04,What three emergent abilities does the survey ...,"In-context learning, instruction following, an...",LLM_survey.pdf,emergent abilities,medium,factual
4,q05,What are the three architectural categories us...,"Encoder-only (e.g., BERT), decoder-only (e.g.,...",LLM_survey.pdf,model architecture,easy,factual


In [ ]:
selected_question_ids = [
    "q01", "q03", "q13", "q21", "q02",
    "q19", "q25", "q30", "q43", "q33",
    "q14", "q27", "q48", "q04", "q16",
    "q17", "q37", "q07", "q23", "q12"
]

selected_questions = que[que['id'].isin(selected_question_ids)]['question'].to_list()

In [ ]:
ground_truth = que[que['id'].isin(selected_question_ids)]['ground_truth_answer'].to_list()

In [ ]:
for experiment_name, results in all_exp.items():
  for result, gt in zip(results, ground_truth):
    result["ground_truth"] = gt

In [ ]:
all_exp['emb_a_recursive'][0].keys()

dict_keys(['question', 'context', 'answer', 'ground_truth'])

## 4. Metric Sanity Check

Before the full evaluation, the four RAGAS metrics were tested on one sample from the `emb_a_semantic` configuration.

In [ ]:
sample = experiments["emb_a_semantic"][0]

In [ ]:
import time

start = time.time()

faithfulness_result = await faithfulness.ascore(
    user_input=sample["question"],
    response=sample["answer"],
    retrieved_contexts=sample["context"]
)

print("Faithfulness:", faithfulness_result.value)
print("Time:", round(time.time() - start, 2), "seconds")

Faithfulness: 0.0
Time: 10.27 seconds


In [ ]:
import time

start = time.time()

answer_relevancy_result = await answer_relevancy.ascore(
    user_input=sample["question"],
    response=sample["answer"],
)

print("Answer Relevancy:", answer_relevancy_result.value)
print("Time:", round(time.time() - start, 2), "seconds")

Answer Relevancy: 0.9221417367845225
Time: 5.14 seconds


In [ ]:
start = time.time()

context_precision_result = await context_precision.ascore(
    user_input=sample["question"],
    retrieved_contexts=sample["context"],
    reference=sample["ground_truth"],
)

print("Context Precision:", context_precision_result.value)
print("Time:", round(time.time() - start, 2), "seconds")

Context Precision: 0.49999999995
Time: 10.41 seconds


In [ ]:
start = time.time()

context_recall_result = await context_recall.ascore(
    user_input=sample["question"],
    retrieved_contexts=sample["context"],
    reference=sample["ground_truth"],
)

print("Context Recall:", context_recall_result.value)
print("Time:", round(time.time() - start, 2), "seconds")

Context Recall: 1.0
Time: 8.32 seconds


## 5. Full RAGAS Evaluation

The complete evaluation covered **20 questions for each retrieval configuration**.

The evaluation was executed with checkpoint-based persistence so that intermediate metric results could be saved to Google Drive. The final comparison below is loaded from the saved result file rather than rerunning the expensive evaluation.

## 6. Final RAGAS Comparison

The saved evaluation results were loaded and reduced to the four RAGAS metrics.  
The mean score for each metric was then calculated for every retrieval configuration.

In [ ]:
import pandas as pd
import json

RESULT_FILE = "/content/drive/MyDrive/RAG_Project/data/ragas_results_with_transparent_estimates.json"

with open(RESULT_FILE, "r") as f:
    results = json.load(f)

metrics = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

In [ ]:
clean_results = {}

for experiment, rows in results.items():

    if experiment.startswith("_"):
        continue

    clean_results[experiment] = pd.DataFrame(rows.values())[metrics]

In [ ]:
avg_df = pd.DataFrame({
    experiment: df.mean()
    for experiment, df in clean_results.items()
}).T

In [ ]:
avg_df.index.name = "Configuration"
avg_df = avg_df.reset_index()

avg_df.round(3)

,Configuration,faithfulness,answer_relevancy,context_precision,context_recall
0,emb_a_semantic,0.362,0.624,0.621,0.6
1,emb_b_semantic,0.325,0.634,0.700,0.6
2,emb_a_recursive,0.400,0.751,0.742,0.7
3,emb_b_recursive,0.275,0.757,0.808,0.7


### Final Retrieval Configuration

Based on the **fully evaluated configurations**, `emb_a_recursive` — **all-MiniLM-L6-v2 + RecursiveCharacterTextSplitter** — was selected as the final retrieval configuration because it provided the strongest overall balance across the four RAGAS metrics among those fully measured.

> **Note:** `emb_b_recursive` contains estimated values for the evaluations that could not be completed. Its row is retained for transparency, but it is not treated as a fully measured configuration.

## Final Note

Due to hardware constraints, the evaluation work was shifted from the local machine to **Google Colab** for GPU-based execution. The final RAGAS comparison was completed from the saved evaluation results, and the project proceeds with the selected retrieval configuration from Phase 4.